In [20]:
import os
import json
import ast
import subprocess
import sys

from pathlib import Path
from typing import TypedDict, Optional, Dict, Any

from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

print("Imports loaded successfully.")

Imports loaded successfully.


In [21]:
fast_model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

strong_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Models initialized.")

Models initialized.


In [22]:
TOOL_REGISTRY = {

    # =========================
    # BASIC
    # =========================

    "calculator": {
        "category": "basic",
        "package": "Python",
        "imports": [],
        "implementation": "Python arithmetic",
        "description": "Perform mathematical calculations."
    },

    "text_processor": {
        "category": "basic",
        "package": "Python",
        "imports": [],
        "implementation": "Python string operations",
        "description": "Process and transform text."
    },


    # =========================
    # PDF
    # =========================

    "pdf_loader": {
        "category": "document",
        "package": "PyMuPDF",
        "imports": ["import pymupdf"],
        "implementation": "pymupdf.open",
        "description": "Load PDF documents."
    },

    "pdf_text_extractor": {
        "category": "document",
        "package": "PyMuPDF",
        "imports": ["import pymupdf"],
        "implementation": "page.get_text",
        "description": "Extract text from PDF pages."
    },

    "ocr": {
        "category": "document",
        "package": "pytesseract",
        "imports": ["import pytesseract"],
        "implementation": "pytesseract.image_to_string",
        "description": "Read scanned documents."
    },

    "document_chunker": {
        "category": "document",
        "package": "langchain-text-splitters",
        "imports": [
            "from langchain_text_splitters import RecursiveCharacterTextSplitter"
        ],
        "implementation": "RecursiveCharacterTextSplitter",
        "description": "Split long documents into chunks."
    },


    # =========================
    # MEMORY / SEARCH
    # =========================

    "embeddings": {
        "category": "retrieval",
        "package": "sentence-transformers",
        "imports": [
            "from sentence_transformers import SentenceTransformer"
        ],
        "implementation": "SentenceTransformer",
        "description": "Generate semantic embeddings."
    },

    "vector_store": {
        "category": "retrieval",
        "package": "faiss-cpu",
        "imports": ["import faiss"],
        "implementation": "faiss.IndexFlatL2",
        "description": "Store and search embeddings."
    },


    # =========================
    # CODING
    # =========================

    "file_reader": {
        "category": "coding",
        "package": "Python",
        "imports": ["from pathlib import Path"],
        "implementation": "Path.read_text",
        "description": "Read project files."
    },

    "file_writer": {
        "category": "coding",
        "package": "Python",
        "imports": ["from pathlib import Path"],
        "implementation": "Path.write_text",
        "description": "Write or modify files."
    },

    "python_executor": {
        "category": "coding",
        "package": "Python",
        "imports": ["import subprocess"],
        "implementation": "subprocess.run",
        "description": "Execute Python code."
    },


    # =========================
    # WEB
    # =========================

    "web_search": {
        "category": "web",
        "package": "requests",
        "imports": ["import requests"],
        "implementation": "HTTP requests",
        "description": "Search or access web information."
    },


    # =========================
    # LLM
    # =========================

    "llm": {
        "category": "core",
        "package": "langchain-groq",
        "imports": [
            "from langchain_groq import ChatGroq"
        ],
        "implementation": "ChatGroq",
        "description": "Generate natural-language responses."
    }
}

print(f"{len(TOOL_REGISTRY)} tools available.")

13 tools available.


In [23]:
class AgentState(TypedDict):

    user_prompt: str

    requirements: str

    complexity: str

    selected_tools: list

    architecture: str

    agent_spec: Dict[str, Any]

    generated_code: Optional[str]

    validation: Optional[Dict[str, Any]]

In [24]:
def analyze_requirements(user_prompt):

    prompt = f"""
Analyze this user request:

{user_prompt}

Determine:

1. Goal
2. Input
3. Output
4. Required capabilities
5. Required tools
6. Complexity

IMPORTANT:

Do NOT add unnecessary technologies.

If the task can be solved with a simple Python
function or one LLM call, classify it as SIMPLE.

If external tools are required, classify it as TOOL.

If multiple tools, memory, retrieval, planning,
or multi-step orchestration are required,
classify it as COMPLEX.

Return JSON:

{{
    "goal": "",
    "input": "",
    "output": "",
    "capabilities": [],
    "required_tools": [],
    "complexity": "SIMPLE | TOOL | COMPLEX"
}}

Only return JSON.
"""

    response = fast_model.invoke(prompt)

    content = response.content.strip()

    if content.startswith("```"):
        content = (
            content
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

    return json.loads(content)

In [25]:
def select_tools(requirements):

    available_tools = []

    for name, info in TOOL_REGISTRY.items():

        available_tools.append({
            "name": name,
            "description": info["description"],
            "category": info["category"]
        })

    prompt = f"""
You are a Tool Selection Agent.

USER REQUIREMENTS:

{json.dumps(requirements, indent=4)}

AVAILABLE TOOLS:

{json.dumps(available_tools, indent=4)}

Rules:

1. Select ONLY tools actually required.
2. Do NOT select tools just because they exist.
3. Prefer the minimum number of tools.
4. SIMPLE tasks should normally use only:
   - llm
   - or no tools.
5. Do NOT select embeddings, FAISS, OCR,
   LangGraph, etc. unless the task actually requires them.
6. Never invent tools.

Return ONLY a JSON list.

Example:

["llm"]

or:

["pdf_loader", "pdf_text_extractor", "llm"]
IMPORTANT REQUIREMENT COVERAGE RULE:

Every explicit user requirement must be mapped to at least
one tool or implementation capability.

For example:

If the user asks for:
"page-level citations"

you MUST select:

"citation_generator"

If the user asks for:
"scanned PDFs"

you MUST select:

"ocr"

If the user asks for:
"PDF question answering"

you MUST select the required PDF ingestion,
retrieval, and LLM tools.

Never omit an explicitly requested capability.
"""

    response = fast_model.invoke(prompt)

    content = response.content.strip()

    if content.startswith("```"):
        content = (
            content
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

    tools = json.loads(content)

    # Safety filter
    tools = [
        tool for tool in tools
        if tool in TOOL_REGISTRY
    ]

    return tools

In [26]:
def generate_architecture(requirements, tools):

    complexity = requirements["complexity"]

    # =====================================
    # SIMPLE
    # =====================================

    if complexity == "SIMPLE":

        return """
Architecture:

User Input
    ↓
LLM
    ↓
Response
"""

    # =====================================
    # TOOL
    # =====================================

    if complexity == "TOOL":

        return f"""
Architecture:

User Input
    ↓
LLM
    ↓
Selected Tools
    ↓
LLM
    ↓
Response

Selected tools:
{tools}
"""

    # =====================================
    # COMPLEX
    # =====================================

    return f"""
Architecture:

User Input
    ↓
Planner
    ↓
Tool Selection
    ↓
Tool Execution
    ↓
Result Processing
    ↓
LLM
    ↓
Final Response

Selected tools:
{tools}

Use LangGraph only if orchestration
actually requires multiple steps.
"""

In [27]:
def create_agent_spec(
    user_prompt,
    requirements,
    complexity,
    tools,
    architecture
):

    return {
        "agent_name": "GeneratedAgent",
        "description": requirements["goal"],
        "goal": requirements["goal"],

        "input": requirements["input"],

        "output": requirements["output"],

        "complexity": complexity,

        "tools": tools,

        "architecture": architecture,

        "model": {
            "provider": "groq",

            "model": (
                "openai/gpt-oss-20b"
                if complexity == "SIMPLE"
                else "openai/gpt-oss-120b"
            ),

            "temperature": 0
        }
    }

In [28]:
def generate_code(agent_spec):

    complexity = agent_spec["complexity"]

    tools = agent_spec["tools"]

    tool_details = []

    for tool in tools:

        info = TOOL_REGISTRY[tool]

        tool_details.append(
            {
                "name": tool,
                "package": info["package"],
                "imports": info["imports"],
                "implementation": info["implementation"],
                "description": info["description"]
            }
        )

    prompt = f"""
You are an expert Python AI Agent Developer.

Generate a Python agent from this specification:

{json.dumps(agent_spec, indent=4)}

TOOLS:

{json.dumps(tool_details, indent=4)}

==================================================
MOST IMPORTANT RULE
==================================================

DO NOT OVERENGINEER THE AGENT.

The generated code must use the SIMPLEST
architecture that satisfies the user's request.

==================================================
SIMPLE AGENT
==================================================

If complexity is SIMPLE:

DO NOT use:

- LangGraph
- FAISS
- embeddings
- vector databases
- OCR
- document chunking
- planners
- multiple agents
- unnecessary classes
- unnecessary abstractions

A simple agent should look approximately like:

from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

def agent(user_input):
    response = model.invoke(user_input)
    return response.content

==================================================
TOOL AGENT
==================================================

If complexity is TOOL:

Use only the required tools.

Do NOT add tools that were not selected.

==================================================
COMPLEX AGENT
==================================================

Use LangGraph only when the task genuinely requires:

- multiple steps
- planning
- multiple tool calls
- state
- branching
- retries
- orchestration

==================================================
PDF EXAMPLE
==================================================

If PDF tools are selected:

Use:

import pymupdf

doc = pymupdf.open(
    stream=file_bytes,
    filetype="pdf"
)

Use actual page extraction:

page.get_text()

Use OCR only when required.

Use embeddings and FAISS only when retrieval
is actually required.

Preserve:

document_id
file_name
page_number
chunk_id

==================================================
IMPORT RULES
==================================================

Use only imports required by selected tools.

Never use:

import fitz

Never use deprecated LangChain APIs.

For Groq:

from langchain_groq import ChatGroq

==================================================
OUTPUT
==================================================

Return ONLY Python code.

No Markdown.

No explanation.

No code fences.

The code must be executable.

Include:

if __name__ == "__main__":

with a minimal test appropriate for the agent.
"""

    model = (
        fast_model
        if complexity == "SIMPLE"
        else strong_model
    )

    response = model.invoke(prompt)

    code = response.content.strip()

    if code.startswith("```"):

        code = (
            code
            .replace("```python", "")
            .replace("```", "")
            .strip()
        )

    return code

In [29]:
def validate_syntax(code):

    try:

        ast.parse(code)

        return True, "Syntax valid."

    except SyntaxError as e:

        return False, (
            f"Syntax error: {e.msg} "
            f"at line {e.lineno}"
        )

In [30]:
APPROVED_IMPORTS = {

    "os",
    "json",
    "subprocess",
    "sys",
    "pathlib",

    "pymupdf",
    "pytesseract",
    "faiss",
    "numpy",
    "requests",

    "langchain_groq",
    "langchain_text_splitters",
    "sentence_transformers",

    "langgraph"
}


def validate_imports(code):

    try:

        tree = ast.parse(code)

    except SyntaxError as e:

        return False, str(e)

    errors = []

    for node in ast.walk(tree):

        if isinstance(node, ast.Import):

            for alias in node.names:

                if alias.name not in APPROVED_IMPORTS:

                    errors.append(
                        f"Unapproved import: "
                        f"{alias.name}"
                    )

        elif isinstance(node, ast.ImportFrom):

            module = node.module or ""

            if module not in APPROVED_IMPORTS:

                errors.append(
                    f"Unapproved module: "
                    f"{module}"
                )

    if errors:

        return False, "\n".join(errors)

    return True, "Imports valid."

In [31]:
def validate_complexity(code, agent_spec):

    complexity = agent_spec["complexity"]

    errors = []

    if complexity == "SIMPLE":

        forbidden = [
            "StateGraph",
            "FAISS",
            "faiss",
            "SentenceTransformer",
            "RecursiveCharacterTextSplitter",
            "pytesseract",
            "pymupdf",
            "index.search",
            "planner"
        ]

        for item in forbidden:

            if item in code:

                errors.append(
                    f"SIMPLE agent contains unnecessary "
                    f"complex component: {item}"
                )

    return (
        False,
        "\n".join(errors)
    ) if errors else (
        True,
        "Complexity is appropriate."
    )

In [32]:
def validate_tools(code, tools):

    errors = []

    implementations = {

        "pdf_loader":
            "pymupdf.open",

        "pdf_text_extractor":
            "get_text",

        "ocr":
            "image_to_string",

        "document_chunker":
            "RecursiveCharacterTextSplitter",

        "embeddings":
            "SentenceTransformer",

        "vector_store":
            "IndexFlatL2",

        "retriever":
            "search",

        "web_search":
            "requests",

        "file_reader":
            "read_text",

        "file_writer":
            "write_text",

        "python_executor":
            "subprocess.run",

        "llm":
            "ChatGroq"
    }

    for tool in tools:

        if tool in implementations:

            required = implementations[tool]

            if required not in code:

                errors.append(
                    f"Tool '{tool}' selected but "
                    f"implementation '{required}' "
                    f"was not found."
                )

    if errors:

        return False, "\n".join(errors)

    return True, "Tools implemented."

In [33]:
def build_agent(user_prompt):

    print("\n" + "=" * 70)
    print("USER PROMPT")
    print("=" * 70)

    print(user_prompt)

    # =====================================
    # 1. Requirements
    # =====================================

    requirements = analyze_requirements(
        user_prompt
    )

    print("\nRequirement Analysis       ✅")

    # =====================================
    # 2. Complexity
    # =====================================

    complexity = requirements["complexity"]

    print(
        f"Complexity                 → {complexity}"
    )

    # =====================================
    # 3. Tools
    # =====================================

    tools = select_tools(
        requirements
    )

    print(
        "Selected Tools             →",
        tools
    )

    # =====================================
    # 4. Architecture
    # =====================================

    architecture = generate_architecture(
        requirements,
        tools
    )

    print(
        "Architecture               ✅"
    )

    # =====================================
    # 5. Agent Specification
    # =====================================

    agent_spec = create_agent_spec(
        user_prompt,
        requirements,
        complexity,
        tools,
        architecture
    )

    print(
        "Agent Specification        ✅"
    )

    # =====================================
    # 6. Generate Code
    # =====================================

    code = generate_code(
        agent_spec
    )

    print(
        "Code Generation            ✅"
    )

    # =====================================
    # 7. Validation
    # =====================================

    for attempt in range(3):

        print(
            f"\nValidation Attempt "
            f"{attempt + 1}/3"
        )

        # Syntax
        ok, msg = validate_syntax(code)

        if not ok:

            print("❌ Syntax:", msg)

            code = generate_code(
                agent_spec
            )

            continue

        print("✅ Syntax")

        # Imports
        ok, msg = validate_imports(code)

        if not ok:

            print("❌ Imports:", msg)

            code = generate_code(
                agent_spec
            )

            continue

        print("✅ Imports")

        # Complexity
        ok, msg = validate_complexity(
            code,
            agent_spec
        )

        if not ok:

            print("❌ Complexity:", msg)

            code = generate_code(
                agent_spec
            )

            continue

        print("✅ Complexity")

        # Tools
        ok, msg = validate_tools(
            code,
            tools
        )

        if not ok:

            print("❌ Tools:", msg)

            code = generate_code(
                agent_spec
            )

            continue

        print("✅ Tools")

        print(
            "\n🎉 AGENT GENERATED SUCCESSFULLY"
        )

        return {
            "agent_spec": agent_spec,
            "code": code,
            "valid": True,
            "attempts": attempt + 1
        }

    return {
        "agent_spec": agent_spec,
        "code": code,
        "valid": False,
        "attempts": 3
    }

In [34]:
def save_agent(result):

    path = Path(
        "generated_agent.py"
    )

    path.write_text(
        result["code"],
        encoding="utf-8"
    )

    print(
        f"\nAgent saved to: {path.absolute()}"
    )

    return path

In [35]:
result = build_agent(
    "Create a simple AI agent that answers user questions."
)

print("\n" + "=" * 70)
print("GENERATED CODE")
print("=" * 70)

print(result["code"])

print("\nVALID:", result["valid"])


USER PROMPT
Create a simple AI agent that answers user questions.

Requirement Analysis       ✅
Complexity                 → SIMPLE
Selected Tools             → ['llm']
Architecture               ✅
Agent Specification        ✅
Code Generation            ✅

Validation Attempt 1/3
✅ Syntax
✅ Imports
✅ Complexity
✅ Tools

🎉 AGENT GENERATED SUCCESSFULLY

GENERATED CODE
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

def agent(user_input: str) -> str:
    response = model.invoke(user_input)
    return response.content

if __name__ == "__main__":
    question = "What is the capital of France?"
    answer = agent(question)
    print(f"Question: {question}\nAnswer: {answer}")

VALID: True


In [36]:
agent_path = save_agent(result)

print("\nRunning generated agent...")
print("=" * 70)

import subprocess
import sys

process = subprocess.run(
    [sys.executable, str(agent_path)],
    capture_output=True,
    text=True,
    timeout=60
)

print("RETURN CODE:", process.returncode)

if process.stdout:
    print("\nOUTPUT:")
    print(process.stdout)

if process.stderr:
    print("\nERROR / WARNING:")
    print(process.stderr)


Agent saved to: c:\Users\hp\OneDrive\Desktop\utsarjan\generated_agent.py

Running generated agent...
RETURN CODE: 0

OUTPUT:
Question: What is the capital of France?
Answer: The capital of France is **Paris**.



In [37]:
pdf_result = build_agent(
    "Create an AI agent that reads PDF files, "
    "answers questions from them, and provides "
    "page-level citations."
)

print("\n" + "=" * 70)
print("PDF AGENT RESULT")
print("=" * 70)

print("Complexity:")
print(pdf_result["agent_spec"]["complexity"])

print("\nSelected Tools:")
print(pdf_result["agent_spec"]["tools"])

print("\nValid:")
print(pdf_result["valid"])

print("\nGenerated Code:")
print(pdf_result["code"])


USER PROMPT
Create an AI agent that reads PDF files, answers questions from them, and provides page-level citations.

Requirement Analysis       ✅
Complexity                 → COMPLEX
Selected Tools             → ['pdf_loader', 'pdf_text_extractor', 'document_chunker', 'embeddings', 'vector_store', 'llm']
Architecture               ✅
Agent Specification        ✅


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kww9p4ntfccv0axn0x8bc7d4` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197476, Requested 4035. Please try again in 10m52.752s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [42]:
import uuid
from pathlib import Path


# ============================================================
# AGENT REGISTRY
# ============================================================

AGENT_REGISTRY = {}


# ============================================================
# GENERATE UNIQUE AGENT ID
# ============================================================

def generate_agent_id(agent_type):

    short_id = uuid.uuid4().hex[:8]

    return f"{agent_type}_{short_id}"


# ============================================================
# CREATE API WRAPPER
# ============================================================

def create_api_wrapper(agent_id, agent_file):

    wrapper_code = f'''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import importlib.util


app = FastAPI(
    title="Generated Agent API",
    description="API for generated agent: {agent_id}"
)


# ============================================================
# LOAD GENERATED AGENT
# ============================================================

spec = importlib.util.spec_from_file_location(
    "generated_agent",
    "{agent_file}"
)

agent_module = importlib.util.module_from_spec(spec)

spec.loader.exec_module(agent_module)


# ============================================================
# REQUEST MODEL
# ============================================================

class AgentRequest(BaseModel):
    prompt: str


# ============================================================
# HEALTH CHECK
# ============================================================

@app.get("/")
def health():

    return {{
        "status": "running",
        "agent_id": "{agent_id}"
    }}


# ============================================================
# RUN AGENT
# ============================================================

@app.post("/agents/{agent_id}/run")
def run_agent(request: AgentRequest):

    try:

        if hasattr(agent_module, "agent"):

            result = agent_module.agent(
                request.prompt
            )

        elif hasattr(agent_module, "run_agent"):

            result = agent_module.run_agent(
                request.prompt
            )

        else:

            raise Exception(
                "Generated agent must contain "
                "agent() or run_agent()."
            )

        return {{
            "agent_id": "{agent_id}",
            "response": str(result)
        }}

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )
'''

    wrapper_path = f"{agent_id}_api.py"

    Path(wrapper_path).write_text(
        wrapper_code,
        encoding="utf-8"
    )

    print(
        f"API wrapper created: {wrapper_path}"
    )

    return wrapper_path


# ============================================================
# REGISTER AGENT
# ============================================================

def register_agent(
    agent_type,
    generated_code
):

    # Generate unique ID
    agent_id = generate_agent_id(
        agent_type
    )

    # Generated agent file
    agent_file = f"{agent_id}.py"

    Path(agent_file).write_text(
        generated_code,
        encoding="utf-8"
    )

    # Create API wrapper
    api_file = create_api_wrapper(
        agent_id,
        agent_file
    )

    # Save in registry
    AGENT_REGISTRY[agent_id] = {

        "agent_id": agent_id,

        "type": agent_type,

        "agent_file": agent_file,

        "api_file": api_file,

        "endpoint":
            f"/agents/{agent_id}/run"
    }

    print("\n" + "=" * 60)
    print("AGENT REGISTERED")
    print("=" * 60)

    print(
        "Agent ID:",
        agent_id
    )

    print(
        "Agent File:",
        agent_file
    )

    print(
        "API File:",
        api_file
    )

    print(
        "Endpoint:",
        f"/agents/{agent_id}/run"
    )

    return AGENT_REGISTRY[agent_id]

In [39]:
def register_agent(
    agent_type,
    generated_code
):

    agent_id = generate_agent_id(
        agent_type
    )

    agent_file = f"{agent_id}.py"

    Path(agent_file).write_text(
        generated_code,
        encoding="utf-8"
    )

    api_file = create_api_wrapper(
        agent_id,
        agent_file
    )

    AGENT_REGISTRY[agent_id] = {

        "agent_id": agent_id,

        "type": agent_type,

        "agent_file": agent_file,

        "api_file": api_file,

        "endpoint":
            f"/agents/{agent_id}/run"
    }

    print("\nAgent registered successfully!")

    print(
        "Agent ID:",
        agent_id
    )

    print(
        "Endpoint:",
        f"/agents/{agent_id}/run"
    )

    return AGENT_REGISTRY[agent_id]

In [43]:
result = build_agent(
    "Create a simple AI agent that answers user questions."
)


USER PROMPT
Create a simple AI agent that answers user questions.

Requirement Analysis       ✅
Complexity                 → SIMPLE
Selected Tools             → ['llm']
Architecture               ✅
Agent Specification        ✅
Code Generation            ✅

Validation Attempt 1/3
✅ Syntax
✅ Imports
✅ Complexity
✅ Tools

🎉 AGENT GENERATED SUCCESSFULLY


In [44]:
simple_agent = register_agent(
    "simple",
    result["code"]
)

API wrapper created: simple_e5b5fea7_api.py

AGENT REGISTERED
Agent ID: simple_e5b5fea7
Agent File: simple_e5b5fea7.py
API File: simple_e5b5fea7_api.py
Endpoint: /agents/simple_e5b5fea7/run


In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENCODE_API_KEY"),
    base_url="https://opencode.ai/zen/v1",
)

response = client.chat.completions.create(
    model="deepseek-v4-flash-free",
    messages=[
        {
            "role": "user",
            "content": "Create a simple Python AI agent."
        }
    ],
)

print(response.choices[0].message.content)

Here's a simple Python AI agent that uses the classic **perceive → decide → act** loop. It’s a chatbot with basic memory and no external dependencies.

```python
import re
import datetime

class SimpleAgent:
    def __init__(self):
        self.name = "Bot"
        self.memory = []

    # Step 1: Get input from the environment/user
    def perceive(self, user_input):
        return user_input.strip().lower()

    # Step 2: Decide what to do based on the input
    def decide(self, perception):
        if re.search(r"\b(hi|hello|hey)\b", perception):
            return "Hello! How can I help you?"

        elif "your name" in perception:
            return f"My name is {self.name}."

        elif "time" in perception:
            now = datetime.datetime.now().strftime("%H:%M")
            return f"The current time is {now}."

        elif "remember" in perception:
            item = perception.replace("remember", "").strip()
            self.memory.append(item)
            return f"OK, I

: 